# 💬 Twitter Sentiment Analysis using Deep Learning
## RNN, LSTM, GRU - Text Classification
**Dataset:** Twitter Entity Sentiment Analysis

Classifies tweets into Positive, Negative, Neutral, and Irrelevant sentiments.

---
## 1. 📦 Import Libraries

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os, pickle, re, string

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, SimpleRNN, LSTM, GRU, Embedding
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, roc_auc_score, roc_curve)

print(f"TensorFlow: {tf.__version__}")


---
## 2. 📊 Dataset Understanding

In [ ]:

train_df = pd.read_csv('Twitter Sentiment Analysis/twitter_training.csv',
                        header=None, names=['id', 'entity', 'sentiment', 'text'])
val_df = pd.read_csv('Twitter Sentiment Analysis/twitter_validation.csv',
                      header=None, names=['id', 'entity', 'sentiment', 'text'])

print("=" * 60)
print("TRAINING DATASET OVERVIEW")
print("=" * 60)
print(f"Shape: {train_df.shape}")
print(f"\nColumns: {list(train_df.columns)}")
print(f"\nData Types:\n{train_df.dtypes}")
print(f"\nMissing Values:\n{train_df.isnull().sum()}")
print(f"\nSentiment Distribution:\n{train_df['sentiment'].value_counts()}")
print(f"\nSample Texts:")
print(train_df[['sentiment', 'text']].head(10))
print(f"\nValidation Shape: {val_df.shape}")

# Visualizations
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
train_df['sentiment'].value_counts().plot(kind='bar', ax=axes[0], 
    color=['#2ecc71', '#e74c3c', '#3498db', '#95a5a6'], edgecolor='black')
axes[0].set_title('Sentiment Distribution (Training)', fontweight='bold')
axes[0].set_ylabel('Count')

# Text length distribution
train_df['text_len'] = train_df['text'].astype(str).apply(len)
train_df['text_len'].hist(bins=50, ax=axes[1], color='#9b59b6', edgecolor='black')
axes[1].set_title('Text Length Distribution', fontweight='bold')
axes[1].set_xlabel('Character Length')
plt.tight_layout()
plt.savefig('twitter_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Entity distribution
plt.figure(figsize=(14, 5))
train_df['entity'].value_counts().head(15).plot(kind='bar', color='#1abc9c', edgecolor='black')
plt.title('Top 15 Entities', fontweight='bold')
plt.tight_layout()
plt.savefig('twitter_entities.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 3. 🔧 Preprocessing

**Text preprocessing steps explained:**
- **Lowercasing:** Normalizes text so "Good" and "good" are treated the same
- **Removing punctuation/special chars:** Reduces noise in the vocabulary
- **Tokenization:** Converts words to integer sequences for neural network input
- **Padding:** Makes all sequences same length (required for batch processing)
- **Label encoding:** Converts sentiment labels to numeric values

In [ ]:

# Drop missing text rows
train_df = train_df.dropna(subset=['text', 'sentiment']).reset_index(drop=True)
val_df = val_df.dropna(subset=['text', 'sentiment']).reset_index(drop=True)

# Clean text function
def clean_text(text):
    """Clean and preprocess text data."""
    text = str(text).lower()  # Lowercase
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)  # Remove URLs
    text = re.sub(r'@\w+', '', text)  # Remove mentions
    text = re.sub(r'#\w+', '', text)  # Remove hashtags
    text = re.sub(r'pic\.\S+', '', text)  # Remove pic links
    text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
    text = re.sub(r'\d+', '', text)  # Remove numbers
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra whitespace
    return text

train_df['clean_text'] = train_df['text'].apply(clean_text)
val_df['clean_text'] = val_df['text'].apply(clean_text)

# Remove empty texts after cleaning
train_df = train_df[train_df['clean_text'].str.len() > 0].reset_index(drop=True)
val_df = val_df[val_df['clean_text'].str.len() > 0].reset_index(drop=True)

print(f"After cleaning - Train: {train_df.shape}, Val: {val_df.shape}")
print(f"\nSample cleaned texts:")
print(train_df[['sentiment', 'clean_text']].head())

# Label encode sentiment
le = LabelEncoder()
y_train = le.fit_transform(train_df['sentiment'])
y_val = le.transform(val_df['sentiment'])
n_classes = len(le.classes_)
print(f"\nClasses: {le.classes_}")

# Tokenize text
MAX_VOCAB = 20000
MAX_LEN = 100

tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token='<OOV>')
tokenizer.fit_on_texts(train_df['clean_text'])

X_train_seq = tokenizer.texts_to_sequences(train_df['clean_text'])
X_val_seq = tokenizer.texts_to_sequences(val_df['clean_text'])

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_val_pad = pad_sequences(X_val_seq, maxlen=MAX_LEN, padding='post', truncating='post')

y_train_cat = to_categorical(y_train, n_classes)
y_val_cat = to_categorical(y_val, n_classes)

vocab_size = min(MAX_VOCAB, len(tokenizer.word_index) + 1)
print(f"\nVocabulary size: {vocab_size}")
print(f"X_train shape: {X_train_pad.shape}")
print(f"X_val shape: {X_val_pad.shape}")


---
## 4. 🏗️ Model Building

Each model uses an **Embedding layer** to learn dense word representations.
The embedding maps each word index to a vector of `EMBED_DIM` dimensions.

In [ ]:

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)
EPOCHS = 30
BATCH = 64
EMBED_DIM = 128


### Simple RNN

In [ ]:
print("\n" + "=" * 50)
print("MODEL 1: Simple RNN")
print("=" * 50)
model_rnn = Sequential([
    Embedding(vocab_size, EMBED_DIM, input_length=MAX_LEN),
    SimpleRNN(128, return_sequences=True),
    Dropout(0.3),
    SimpleRNN(64),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(n_classes, activation='softmax')
])
model_rnn.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model_rnn.summary()
history_rnn = model_rnn.fit(X_train_pad, y_train_cat, epochs=EPOCHS, batch_size=BATCH,
                            validation_data=(X_val_pad, y_val_cat), callbacks=[early_stop], verbose=1)


### LSTM

In [ ]:
print("\n" + "=" * 50)
print("MODEL 2: LSTM")
print("=" * 50)
model_lstm = Sequential([
    Embedding(vocab_size, EMBED_DIM, input_length=MAX_LEN),
    LSTM(128, return_sequences=True),
    Dropout(0.3),
    LSTM(64),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(n_classes, activation='softmax')
])
model_lstm.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model_lstm.summary()
history_lstm = model_lstm.fit(X_train_pad, y_train_cat, epochs=EPOCHS, batch_size=BATCH,
                              validation_data=(X_val_pad, y_val_cat), callbacks=[early_stop], verbose=1)


### GRU

In [ ]:
print("\n" + "=" * 50)
print("MODEL 3: GRU")
print("=" * 50)
model_gru = Sequential([
    Embedding(vocab_size, EMBED_DIM, input_length=MAX_LEN),
    GRU(128, return_sequences=True),
    Dropout(0.3),
    GRU(64),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(n_classes, activation='softmax')
])
model_gru.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model_gru.summary()
history_gru = model_gru.fit(X_train_pad, y_train_cat, epochs=EPOCHS, batch_size=BATCH,
                            validation_data=(X_val_pad, y_val_cat), callbacks=[early_stop], verbose=1)


---
## 5. 📈 Training Curves

In [ ]:

models_info = {'RNN': history_rnn, 'LSTM': history_lstm, 'GRU': history_gru}

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for i, (name, hist) in enumerate(models_info.items()):
    axes[0, i].plot(hist.history['accuracy'], label='Train', linewidth=2)
    axes[0, i].plot(hist.history['val_accuracy'], label='Val', linewidth=2)
    axes[0, i].set_title(f'{name} - Accuracy', fontweight='bold')
    axes[0, i].legend()
    axes[1, i].plot(hist.history['loss'], label='Train', linewidth=2)
    axes[1, i].plot(hist.history['val_loss'], label='Val', linewidth=2)
    axes[1, i].set_title(f'{name} - Loss', fontweight='bold')
    axes[1, i].legend()
plt.suptitle('Training vs Validation Curves', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('twitter_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 6. 📋 Evaluation

In [ ]:

def evaluate_cls(model, X_te, y_te_cat, name):
    y_pred_prob = model.predict(X_te)
    y_pred = np.argmax(y_pred_prob, axis=1)
    y_true = np.argmax(y_te_cat, axis=1)
    
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    try:
        auc = roc_auc_score(y_te_cat, y_pred_prob, multi_class='ovr', average='weighted')
    except:
        auc = 0.0
    
    print(f"\n{'=' * 40}")
    print(f"  {name}: Acc={acc:.4f}, F1={f1:.4f}, AUC={auc:.4f}")
    print(f"{'=' * 40}")
    print(classification_report(y_true, y_pred, target_names=le.classes_))
    
    return {'Model': name, 'Accuracy': acc, 'Precision': prec, 'Recall': rec,
            'F1-Score': f1, 'ROC-AUC': auc, 'y_pred': y_pred, 'y_pred_prob': y_pred_prob, 'y_true': y_true}

results = [
    evaluate_cls(model_rnn, X_val_pad, y_val_cat, 'RNN'),
    evaluate_cls(model_lstm, X_val_pad, y_val_cat, 'LSTM'),
    evaluate_cls(model_gru, X_val_pad, y_val_cat, 'GRU')
]

# Confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, r in enumerate(results):
    cm = confusion_matrix(r['y_true'], r['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=le.classes_, yticklabels=le.classes_)
    axes[i].set_title(f"{r['Model']} Confusion Matrix", fontweight='bold')
plt.tight_layout()
plt.savefig('twitter_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

# ROC curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, r in enumerate(results):
    for c in range(n_classes):
        fpr, tpr, _ = roc_curve(y_val_cat[:, c], r['y_pred_prob'][:, c])
        axes[i].plot(fpr, tpr, label=le.classes_[c], linewidth=2)
    axes[i].plot([0, 1], [0, 1], 'k--')
    axes[i].set_title(f"{r['Model']} ROC", fontweight='bold')
    axes[i].legend(fontsize=8)
plt.tight_layout()
plt.savefig('twitter_roc.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 7. 📊 Comparative Analysis

In [ ]:

comp = pd.DataFrame([{k: v for k, v in r.items() if k not in ['y_pred', 'y_pred_prob', 'y_true']}
                     for r in results]).set_index('Model')
print("\n" + "=" * 60)
print("MODEL COMPARISON")
print("=" * 60)
print(comp.round(4))

comp[['Accuracy', 'F1-Score', 'ROC-AUC']].plot(kind='bar', figsize=(10, 5), edgecolor='black')
plt.title('Sentiment Analysis Model Comparison', fontweight='bold')
plt.xticks(rotation=0)
plt.ylim(0, 1.1)
plt.tight_layout()
plt.savefig('twitter_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

best = comp['Accuracy'].idxmax()
print(f"\n🏆 Best model: {best} ({comp.loc[best, 'Accuracy']:.4f})")

print("""
COMPARATIVE ANALYSIS:
- For text/NLP, LSTM/GRU significantly outperform Simple RNN due to their ability
  to capture long-range dependencies in sentences.
- Simple RNN suffers from vanishing gradients - forgets early words in longer texts.
- LSTM's forget gate explicitly controls what information to retain/discard.
- GRU is more parameter-efficient than LSTM, often performing comparably.
- The 4-class imbalance (Irrelevant class) affects all models similarly.
- Embedding layer quality is crucial - pre-trained embeddings (GloVe) could improve results.
""")


---
## 8. 💾 Model Saving

In [ ]:

os.makedirs('saved_models', exist_ok=True)
model_rnn.save('saved_models/twitter_rnn_model.keras')
model_lstm.save('saved_models/twitter_lstm_model.keras')
model_gru.save('saved_models/twitter_gru_model.keras')

with open('saved_models/twitter_tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)
with open('saved_models/twitter_label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)
with open('saved_models/twitter_config.pkl', 'wb') as f:
    pickle.dump({'max_len': MAX_LEN, 'max_vocab': MAX_VOCAB}, f)

comp.to_csv('saved_models/twitter_comparison_results.csv')
print("✅ All models saved! 🎉 Twitter Sentiment Notebook Complete!")
